# Probar un endpoint de Databricks Model Serving

Este notebook realiza una invocación REST a un endpoint de Databricks usando el formato `dataframe_split` y muestra las predicciones devueltas.

El token se obtiene desde Databricks Secrets y nunca se imprime. Antes de ejecutar, configura los widgets de la primera celda y verifica que la identidad tenga permiso para invocar el endpoint.

## Requisitos

- Un endpoint activo de Databricks Model Serving.
- Un secret scope con un token válido.
- Permiso de invocación sobre el endpoint.
- El modelo debe aceptar las cuatro columnas Iris usadas en este ejemplo.

Las librerías `requests` y `pandas` suelen estar disponibles en Databricks Runtime. Si el cluster no las tiene, instala `requests` y `pandas` como librerías del cluster o ejecuta `%pip install requests pandas` en una celda separada y reinicia Python antes de continuar.

In [0]:
import os
from iris_mlflow_utils import load_file_config

SERVING_CONFIG = load_file_config().get("serving", {})
HOST = os.getenv("DATABRICKS_HOST", SERVING_CONFIG.get("host", "")).strip()
ENDPOINT_NAME = os.getenv("DATABRICKS_ENDPOINT_NAME", SERVING_CONFIG.get("endpoint_name", "")).strip()
SECRET_SCOPE = os.getenv("DATABRICKS_SECRET_SCOPE", SERVING_CONFIG.get("secret_scope", "")).strip()
SECRET_KEY = os.getenv("DATABRICKS_SECRET_KEY", SERVING_CONFIG.get("secret_key", "databricks-token")).strip()

if not HOST or "<workspace>" in HOST:
    raise ValueError("Configura DATABRICKS_HOST con el workspace real.")
if not ENDPOINT_NAME:
    raise ValueError("Configura DATABRICKS_ENDPOINT_NAME.")
if not SECRET_SCOPE or not SECRET_KEY:
    raise ValueError("Configura el secret scope y la secret key del token.")

In [0]:
import json
import pandas as pd
import requests

TIMEOUT_SEGUNDOS = int(os.getenv("DATABRICKS_TIMEOUT_SECONDS", SERVING_CONFIG.get("timeout_seconds", 60)))

# Databricks Secrets devuelve el valor del token sin que sea necesario
# escribirlo en el notebook. No lo imprimas ni lo registres en logs.
try:
    TOKEN = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
except Exception as error:
    raise RuntimeError("No se pudo obtener el token desde Databricks Secrets.") from error
if not TOKEN:
    raise ValueError("No se pudo obtener un token válido desde Secrets ni desde el notebook.")

host_limpio = HOST.removeprefix("https://").removeprefix("http://").rstrip("/")
endpoint_limpio = ENDPOINT_NAME.strip("/")
URL = f"https://{host_limpio}/serving-endpoints/{endpoint_limpio}/invocations"
HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}

print(f"Endpoint que se consultará: {URL}")
print(f"Timeout configurado: {TIMEOUT_SEGUNDOS} segundos")
print("Token cargado desde Databricks Secrets: sí (valor oculto)")

In [0]:
columnas = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]
datos = [
    [5.1, 3.5, 1.4, 0.2],
    [6.4, 3.2, 4.5, 1.5],
    [6.7, 3.1, 5.6, 2.4],
]

payload = {
    "dataframe_split": {
        "columns": columnas,
        "data": datos,
    }
}

# Este es el payload que se enviará. No contiene el token.
print(json.dumps(payload, indent=2))

dataframe_entrada = pd.DataFrame(datos, columns=columnas)


## Invocación REST

La respuesta exitosa debe contener una clave `predictions`. Si Databricks devuelve un error HTTP, esta celda muestra el código y el texto devuelto para facilitar el diagnóstico.

In [0]:
try:
    respuesta = requests.post(URL, headers=HEADERS, json=payload, timeout=TIMEOUT_SEGUNDOS)
except requests.Timeout as error:
    raise RuntimeError(
        f"La solicitud superó el timeout de {TIMEOUT_SEGUNDOS} segundos."
    ) from error
except requests.ConnectionError as error:
    raise RuntimeError(
        "No se pudo conectar con Databricks. Revisa el host y la red del cluster."
    ) from error

print(f"Código HTTP: {respuesta.status_code}")
if not respuesta.ok:
    raise RuntimeError(
        f"Databricks devolvió HTTP {respuesta.status_code}: {respuesta.text}"
    )

try:
    respuesta_json = respuesta.json()
except ValueError as error:
    raise RuntimeError(
        f"La respuesta no es JSON válido. Texto devuelto: {respuesta.text}"
    ) from error

if not isinstance(respuesta_json, dict) or "predictions" not in respuesta_json:
    raise RuntimeError("La respuesta no contiene la clave 'predictions'.")

predicciones = respuesta_json["predictions"]
if not isinstance(predicciones, list):
    raise RuntimeError("La clave 'predictions' no contiene una lista.")
if len(predicciones) != len(datos):
    raise RuntimeError(
        f"Se recibieron {len(predicciones)} predicciones para {len(datos)} filas."
    )

print("Respuesta JSON recibida:")
print(json.dumps(respuesta_json, indent=2, ensure_ascii=False))
print(f"Predicciones: {predicciones}")

In [0]:
resultado = dataframe_entrada.copy()
resultado["prediccion"] = predicciones
display(resultado)

## Diagnóstico rápido

- **401**: el token es inválido, expiró o el secret apunta a otra credencial.
- **403**: la identidad del token no tiene permiso para invocar el endpoint.
- **404**: revisa el workspace y el nombre exacto del endpoint.
- **429**: se alcanzó un límite de solicitudes; espera y reintenta con backoff.
- **500/503**: revisa el estado, logs y capacidad del endpoint.
- **400**: revisa que las columnas, el orden y los tipos coincidan con la firma del modelo.